In [ ]:
from typing import Union
import pandas as pd
import numpy as np

import pymc as pm
import pymc.sampling.jax as pmjax
import pytensor.tensor as pt
import arviz as az
import arviz_plots as azp
import xarray as xr

import matplotlib.pyplot as plt

RANDOM_SEED = 694973
np.random.seed(RANDOM_SEED)

import warnings
warnings.filterwarnings("ignore", category=UserWarning)

# for reproducibility
print("pandas: "+pd.__version__)
print("numpy: "+np.__version__)
print("pymc: "+pm.__version__)
print("arviz: "+az.__version__)

pandas: 3.0.1
numpy: 2.4.3
pymc: 6.2.0
arviz: 1.2.0


In [4]:
df = pd.read_csv('data/projection_data.csv')
df.head()

,player_full_id,SEASON,shot_type,PLAYER_ID,PLAYER,player_age,age_z,HEIGHT_INCHES,height_z,birthdate,posit,attempts,actual_makes,expected_makes,offset
0,A.J. Lawson (1630639),2022-23,DUNK,1630639.0,A.J. Lawson,23.041752,-0.976176,78.0,-0.097023,2000-07-15,G,6,5,4.606923,1.196045
1,A.J. Lawson (1630639),2022-23,JUMPER,1630639.0,A.J. Lawson,23.041752,-0.976176,78.0,-0.097023,2000-07-15,G,27,12,10.200240,-0.498953
2,A.J. Lawson (1630639),2022-23,LAYUP,1630639.0,A.J. Lawson,23.041752,-0.976176,78.0,-0.097023,2000-07-15,G,11,5,6.662227,0.429093
3,A.J. Lawson (1630639),2023-24,DUNK,1630639.0,A.J. Lawson,24.043806,-0.740877,78.0,-0.097023,2000-07-15,G,13,12,9.504221,1.000180
4,A.J. Lawson (1630639),2023-24,JUMPER,1630639.0,A.J. Lawson,24.043806,-0.740877,78.0,-0.097023,2000-07-15,G,62,15,23.988975,-0.460282


In [5]:
def define_index(data: pd.DataFrame, label: str) -> Union[np.array, dict]:
    """Defines an index variable starting at 0. 
    Args:
        data (pd.DataFrame): dataframe with values to index
        label (str): column name user wishes to index in string format

    Returns:
        Union[
            np.array: indexed values
            dict: dictionary mapping the input values and their indexed values
            ]
    """
    unq_ids = data[label].astype(str).unique()
    lookup = {v: i for i, v in enumerate(unq_ids)}
    index_vals = data[label].astype(str).map(lookup).values
    return index_vals, lookup

In [6]:
# These are some generic helper functions to help sample for the model
def sample(model: pm.Model, draws: int = 2000, tune: int = 2000, chains: int = 4, target_accept: float = 0.99, random_seed: int = RANDOM_SEED, path:str = 'temp.nc', **kwargs):
    """
    Fit model using MCMC.

    Parameters
    ----------
    model: pm.Model
        PyMC model object.
    draws : int
        Number of draws to keep from the sampling process.
    tune : int
        Number of tuning steps to take before sampling.
    chains : int
        Number of chains to sample.
    target_accept : float
        Target acceptance probability for step size adaptation.
    random_seed : int
        Seed for randomness.
    """
    with model:
        trace = pmjax.sample_numpyro_nuts(
            draws=draws,
            tune=tune,
            chains=chains,
            target_accept=target_accept,
            random_seed=random_seed,
            idata_kwargs={"log_likelihood": False}
        )
    try:
        az.to_netcdf(trace, path)
    except Exception as e:
        print(f"Error saving trace to {path}: {e}")
    return trace

def compute_log_likelihood(model: pm.Model, trace: az.InferenceData) -> None:
    """Wrapper to compute elemwise log_likelihood of model given InferenceData with posterior group
    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling
    """
    with model:
        pm.compute_log_likelihood(trace)
    return None

def sample_posterior_pred(model: pm.Model, trace: az.InferenceData) -> az.InferenceData:
    """Generates samples from the posterior predictive distribution for model checks

    Args:
        model (pm.Model): A PyMC model object
        trace (az.InferenceData): Results from sampling

    Returns:
        az.InferenceData: An ArviZ InferenceData object containing the posterior predictive samples.
    """
    with model:
        spp = pm.sample_posterior_predictive(
            trace,
            extend_inferencedata=True,
            random_seed=RANDOM_SEED,
        )
    return spp

In [9]:
df["player_idx"], player_lookup = define_index(df, "player_full_id")
player_idx_vals = df['player_idx'].values.astype("int32")
n_obs = len(df)

df['season_idx'], season_lookup = define_index(df, "SEASON")
season_idx_vals = df['season_idx'].values.astype("int32")

df['shot_type_idx'], shot_type_lookup = define_index(df, "shot_type")
shot_type_idx_vals = df['shot_type_idx'].values.astype("int32")

df['position_idx'], position_lookup = define_index(df, "posit")
position_idx_vals = df['position_idx'].values.astype("int32")

target = df['actual_makes'].values
offset_vals = df['offset'].values
age_z_vals = df['age_z'].values
height_z_vals = df['height_z'].values
attempts_vals = df['attempts'].values
expected_makes_vals = df['expected_makes'].values

In [ ]:
coords = {
    "obs_id": np.arange(len(target)),
    "shot_type":shot_type_lookup.keys(),
    "player": player_lookup.keys(),
    "season" : ['2021/2022','2022/2023','2023/2024','2024/2025','2025/2026'],
    "season_walk" : ['2022/2023','2023/2024','2024/2025','2025/2026'],
    }

with pm.Model(coords=coords) as model: # mutable data
    shots_made = pm.Data("shots_made", target, dims=("obs_id",))
    offset_data = pm.Data("offset_data", offset_vals, dims=("obs_id",))
    attempts_data = pm.Data("attempts_data", attempts_vals, dims=("obs_id",))
    expected_makes_data = pm.Data("expected_makes_data", expected_makes_vals, dims=("obs_id",))
    player_idx = pm.Data("player_idx", player_idx_vals, dims=("obs_id",))
    season_idx = pm.Data("season_idx", season_idx_vals, dims=("obs_id",))
    shot_type_idx = pm.Data("shot_type_idx", shot_type_idx_vals, dims=("obs_id",))

    # baseline
    offset_weight = pm.Normal("offset_weight", mu=1.0, sigma=0.25)
    shot_type_intercept = pm.Normal("shot_type_intercept", mu=0, sigma=1, dims=("shot_type",))
    # career talent
    sigma_career = pm.HalfNormal("sigma_career", sigma=1.0, dims=("shot_type",))
    player_mean_dev = pm.Normal("player_mean_dev", mu=0.0, sigma=1.0, dims=("player", "shot_type"))
    player_mean_centered = player_mean_dev - player_mean_dev.mean(axis=0)
    player_mean = pm.Deterministic(
        "player_mean",
        player_mean_centered * sigma_career[None, :], dims=("player", "shot_type")
        )
    # walk
    sigma_walk = pm.HalfNormal("sigma_walk", sigma=1.0, dims=("shot_type",))
    player_innovations = pm.Normal(
        "player_innovations",
        mu=0.0, sigma=1.0,
        dims=("season_walk", "player", "shot_type")
    )
    raw_walk = pt.concatenate([
       pt.zeros((1, len(coords["player"]), len(coords["shot_type"]))),
       pt.cumsum(player_innovations, axis=0)and
    ], axis=0)
    player_walk_centered = raw_walk - raw_walk.mean(axis=0, keepdims=True)
    player_walk = player_walk_centered * sigma_walk[None, None, :]
    # player effect
    player_effect = pm.Deterministic(
        "player_effect",
        player_mean[None, :, :] + player_walk, dims=("season", "player", "shot_type")
        )
    player_contribution = player_effect[season_idx, player_idx, shot_type_idx]

    # expected value
    theta = shot_type_intercept[shot_type_idx] + offset_weight*offset_data + player_contribution
    pm.Binomial("obs", n=attempts_data, p=pm.math.sigmoid(theta), observed=shots_made)

In [ ]:
trace = sample(model, tune=1000, draws=1000, target_accept=0.95)